## Softmax Vision transformer trained on Structured labeled data (for finetune) and is used to compare the baseline accuracy
### With subset of 10000 samples and num_epochs = 20

In [ ]:
import torch
import torch.nn as nn
import h5py
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.nn import CrossEntropyLoss
import math
from tqdm.notebook import tqdm, trange
from sklearn.metrics import precision_score
import torchvision.transforms as T
import torchvision.transforms.functional as f
import random

In [ ]:
class JetDataset(Dataset):
    def __init__(self, path, augment=False):
        self.file = h5py.File(path, "r")
        self.images = self.file["jet"]
        self.labels = self.file["Y"]
        self.augment = augment
        self.aug_transforms = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=15)  # small rotation is safer
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        image = image.permute(2, 0, 1)
        if self.augment:
            image = self.aug_transforms(image)
        image = (image - image.mean()) / (image.std() + 1e-6)
        return image, label

In [ ]:
path = "/content/drive/MyDrive/CERN/Dataset_Specific_labelled.h5"

In [ ]:
total_size = len(JetDataset(path))
train_size = int(0.7 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

base_dataset = JetDataset(path, augment=False)

train_idx, val_idx, test_idx = random_split(
    range(total_size),
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
from torch.utils.data import Subset
train_dataset = Subset(JetDataset(path, augment=True), train_idx.indices)
val_dataset   = Subset(JetDataset(path, augment=False), val_idx.indices)
test_dataset  = Subset(JetDataset(path, augment=False), test_idx.indices)

In [ ]:
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

In [ ]:
class patchEmbedding(nn.Module):
  def __init__(self,patch_size,in_channels,embed_dim):
    super().__init__()
    self.proj = nn.Conv2d(
        in_channels = in_channels,
        out_channels = embed_dim,
        stride = patch_size,
        kernel_size = patch_size
    )
  def forward(self,x):
    ## assuming the shape of x = (batch size,channels,height,width)
    x = self.proj(x) # (batch_size,200,25,25)
    #print("proj",x.shape)
    x = x.flatten(2,-1) # (batch,625,200)
    #print("flatten",x.shape)
    x = x.transpose(1,2) # (batch_size,625,200)
    #print("transpose",x.shape)
    return x

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, patches, embed_dim):
        super().__init__()
        self.d_k = embed_dim
        self.patches = patches + 1 # to counter the cls token error
        pe = torch.zeros(self.patches, self.d_k)
        div = torch.exp(
            torch.arange(0, self.d_k, 2).float() * -(math.log(1000.0) / self.d_k)
        )
        position = torch.arange(0, self.patches).float().unsqueeze(1)
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class MultiHeadAttention(nn.Module): # for base line test
  def __init__(self,seq_len,embedding_dim,heads,head_dim,num_chunks,chunk_size):
    super().__init__()
    #assert seq_len == math.floor(seq_len / chunk_size) *chunk_size
    assert embedding_dim == heads * head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.chunk_size = chunk_size
    self.num_chunks = math.ceil(seq_len / chunk_size)
    self.Q = nn.Linear(embedding_dim,embedding_dim)
    self.K = nn.Linear(embedding_dim,embedding_dim)
    self.V = nn.Linear(embedding_dim,embedding_dim)

  def forward(self,x):
    B,L,E =  x.shape
    #print(type(L))
    q = (self.Q(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    k = (self.K(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    v = (self.V(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    d_k = q.shape[-1]
    attention_scores = F.softmax((q @ k.transpose(-1,-2)/d_k),dim=-1)
    out = attention_scores @ v
    out = out.transpose(1, 2).contiguous().reshape(B, L, E)
    return out

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self):
    super().__init__()
    self.eps = 0.01
    self.alpha = nn.Parameter(torch.ones(1))
    self.beta = nn.Parameter(torch.ones(1))
    pass
  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    std = x.std(dim=-1,keepdim=True)
    return (self.alpha * ((x - mean)/(std+self.eps))) + self.beta

In [ ]:
class FeedForwardlayer(nn.Module):
  def __init__(self,patches,embed_dim):
    super().__init__()
    self.dim = 348
    self.fc_1 = nn.Linear(embed_dim,self.dim)
    self.fc_2 = nn.Linear(self.dim,embed_dim)
    self.relu = nn.ReLU()
  def forward(self,x):
    return self.fc_2(self.relu(self.fc_1(x)))

In [ ]:
class ResidualBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm = LayerNorm()
  def forward(self,X,sublayer):
    #print("in residual ")
    return X + sublayer(self.layer_norm(X)) # ada and norm
    ## here the sublayer is normed first then add

In [ ]:
class EncoderBlock(nn.Module):
  def __init__(self,attention:MultiHeadAttention,fnn:FeedForwardlayer,patches,embed_dim):
    super().__init__()
    self.attention = attention
    self.fnn = fnn
    self.patches = patches
    self.embed_dim = embed_dim
    self.residual = nn.ModuleList([ResidualBlock() for _ in range(2)]) # we want and 2 residual layer connected
  def forward(self,X):
    X = self.residual[0](X,lambda x:self.attention(X)) ## here the attention is second layer (sublayer) and the X is the input which is going to add
    X = self.residual[1](X,lambda x:self.fnn(X))
    return X

In [ ]:
class ViTClassifier(nn.Module):
    def __init__(self,embeded_dim,num_classes):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(embeded_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(2048, num_classes)
        )

    def forward(self, x):
        return self.head(x)

In [ ]:
class ViT(nn.Module):
    def __init__(self, patches=625,patch_size=5, embed_dim=200, num_classes=2,in_channels=8):
        super().__init__()
        self.patch_embedding = patchEmbedding(patch_size,in_channels, embed_dim)
        self.positional = PositionalEncoding(patches, embed_dim)
        self.encoder = encoder
        self.classifier = classifier
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x):
        x = self.patch_embedding(x)
        #print(x.shape)
        #print("cls",cls_token.shape)
        #print(x.shape)
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), x], dim=1)
        #print(x.shape)
        x = self.positional(x)
        x = self.encoder(x)
        cls_features = x[:, 0, :]  # Take CLS token
        return self.classifier(cls_features)


In [ ]:
patches = 625
embed_dim =  200
num_classes = 2
patch_size = 5
heads = 4
head_dim = 50
num_chunks = 625
chunk_size = 5
in_channels = 8
patch_embedding = patchEmbedding(patch_size,in_channels,embed_dim)
positional = PositionalEncoding(patches,embed_dim)
attention = MultiHeadAttention(patches,embed_dim,heads,head_dim,num_chunks,chunk_size)
residual = ResidualBlock()
fnn = FeedForwardlayer(patches,embed_dim)
encoder = EncoderBlock(attention,fnn,patches,embed_dim)
classifier = ViTClassifier(embed_dim,num_classes)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = ViT(patches, patch_size, embed_dim, num_classes=2,in_channels=8).to(device)

optimizer = optim.AdamW(model.parameters(), lr=5e-4,weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
epochs = 15
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.2)

In [ ]:
from tqdm.notebook import tqdm, trange
from sklearn.metrics import precision_score, recall_score, f1_score

train_losses, val_losses = [], []
train_accs, val_accs = [], []
train_precs, val_precs = [], []
train_recalls, val_recalls = [], []
train_f1s, val_f1s = [], []

best_val_acc = 0
early_stop_patience = 5
early_stop_counter = 0

for epoch in trange(epochs, desc="Training Progress"):
    model.train()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    train_bar = tqdm(train_loader, desc=f"Train {epoch+1}", leave=False)

    for image, label in train_bar:
        image = image.to(device)
        label = label.to(device).squeeze(1)

        optimizer.zero_grad()
        pred = model(image)

        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(pred, 1)

        total += label.size(0)
        correct += (predicted == label).sum().item()

        all_preds.extend(predicted.detach().cpu().numpy())
        all_labels.extend(label.detach().cpu().numpy())

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100*correct/total:.2f}%"
        })

    train_loss = total_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_prec = precision_score(all_labels, all_preds, zero_division=0)
    train_recall = recall_score(all_labels, all_preds, zero_division=0)
    train_f1 = f1_score(all_labels, all_preds, zero_division=0)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_precs.append(train_prec)
    train_recalls.append(train_recall)
    train_f1s.append(train_f1)
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    val_preds, val_labels = [], []

    val_bar = tqdm(val_loader, desc=f"Val {epoch+1}", leave=False)

    with torch.no_grad():
        for image, label in val_bar:
            image = image.to(device)
            label = label.to(device).squeeze(1)

            pred = model(image)
            loss = criterion(pred, label)

            val_loss += loss.item()

            _, predicted = torch.max(pred, 1)

            val_total += label.size(0)
            val_correct += (predicted == label).sum().item()

            val_preds.extend(predicted.detach().cpu().numpy())
            val_labels.extend(label.detach().cpu().numpy())

    val_loss_avg = val_loss / len(val_loader)
    val_acc = 100. * val_correct / val_total
    val_prec = precision_score(val_labels, val_preds, zero_division=0)
    val_recall = recall_score(val_labels, val_preds, zero_division=0)
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)

    val_losses.append(val_loss_avg)
    val_accs.append(val_acc)
    val_precs.append(val_prec)
    val_recalls.append(val_recall)
    val_f1s.append(val_f1)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        early_stop_counter = 0

        torch.save(model.state_dict(), "best_model_weights.pth")

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_acc": val_acc
        }, "best_model_full.pth")

    else:
        early_stop_counter += 1

    scheduler.step(val_loss)
    print(f"\nEpoch {epoch+1:02d}/{epochs}")
    print(f"Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%, Prec={train_prec:.3f}, Recall={train_recall:.3f}, F1={train_f1:.3f}")
    print(f"Val:   Loss={val_loss_avg:.4f}, Acc={val_acc:.2f}%, Prec={val_prec:.3f}, Recall={val_recall:.3f}, F1={val_f1:.3f}, Best={best_val_acc:.2f}%")
    print(f"LR: {scheduler.get_last_lr()[0]:.2e}")
    print("-" * 60)
    if early_stop_counter >= early_stop_patience:
        print("Early stopping triggered")
        break

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))

ax1.plot(train_losses, label='Train Loss', linewidth=2)
ax1.plot(val_losses, label='Val Loss', linewidth=2)
ax1.set_title('Loss')
ax1.legend()

ax2.plot(train_accs, label='Train Acc', linewidth=2)
ax2.plot(val_accs, label='Val Acc', linewidth=2)
ax2.set_title('Accuracy (%)')
ax2.legend()

ax3.plot(train_precs, label='Train Prec', linewidth=2)
ax3.plot(val_precs, label='Val Prec', linewidth=2)
ax3.set_title('Precision')
ax3.legend()

plt.tight_layout()
plt.savefig('vit_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nFinal Best Val Acc: {best_val_acc:.2f}%")
print(f"Best model saved: best_vit_jet_model.pth")
